# MoE generator + reconstruction / distillation: experiments

Baseline MPLMM generates a missing modality with **one fixed function** for every sample. Here:

`x̂_m = G_base(x)_m + tanh(gate_m) · Σ_k p_k(x) · expert_k(x)_m`

* **H2 (MoE):** K = 3 light experts with per-sample router weights `p_k` (1/3 each at the start); the gate starts at 0, so training starts from the baseline.
* **H1 (extra losses, training only):**
  * **rec** (reconstruction): the generated modality should look like the *real* one, which is still in the training batch.
  * **kd** (distillation): the model's representation with a modality missing should match its own representation of the *complete* sample.

| Run | moe | λ_rec | λ_kd |
|---|---|---|---|
| `moe` (H2 alone) | 1 | 0 | 0 |
| `moe + rec + kd` (H1 + H2, proposed) | 1 | 0.5 | 0.5 |

Both are compared with the final **baseline table**, the **RAG** results and the **paper**. All logic lives in `moemm/` and `scripts/`; see `README.md`.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "moemm").is_dir() else Path.cwd() / "MoEPrompt"
RUNS_CSV = ROOT / "results" / "runs.csv"
CASES = ["{a}", "{v}", "{t}", "{a,v}", "{a,t}", "{v,t}", "Avg."]
DATASETS = ["mosi", "sims", "iemocap", "mosei"]  # fastest first
NAMES = {"mosi": "MOSI", "sims": "CH-SIMS", "iemocap": "IEMOCAP", "mosei": "MOSEI"}
SEED = 1111
GRID = [  # (moe, lambda_rec, lambda_kd)
    (1, 0.0, 0.0),  # moe
    (1, 0.5, 0.5),  # moe + rec + kd
]


def run(script, *args):
    """Run one of the scripts and stream its output into the notebook."""
    cmd = [sys.executable, str(ROOT / "scripts" / script), *map(str, args)]
    subprocess.run(cmd, check=True, cwd=ROOT, env={**os.environ, "PYTHONWARNINGS": "ignore"})


def run_name(ds, moe, rec, kd, seed=SEED, k=3):
    core = f"moe_k{k}" if moe else "base"
    return f"{ds}_{core}_rec{rec:g}_kd{kd:g}_seed{seed}"


def done(name):
    return RUNS_CSV.exists() and name in set(pd.read_csv(RUNS_CSV)["run"])


print("MoEPrompt root:", ROOT)

MoEPrompt root: /home/cse/Desktop/MLProjectVidit/MoEPrompt


## 1. Sanity checks (no training, about 5 min)

Each must end with **ALL CHECKS PASSED**. The Stage 0 diagnosis is already saved in `results/diagnose_*.csv` and shown below (oracle = real features instead of generated ones = the ceiling).

In [2]:
for ds in DATASETS:
    run("sanity_checks.py", "--dataset", ds)

diag = pd.concat([pd.read_csv(ROOT / "results" / f"diagnose_{ds}_test.csv") for ds in DATASETS])
diag[["dataset", "variant", "Avg. ACC", "Avg. F1"]]

Missing key(s) in state_dict :proj_l.weight
Missing key(s) in state_dict :proj_a.weight
Missing key(s) in state_dict :proj_v.weight
Missing key(s) in state_dict :out_layer.weight
Missing key(s) in state_dict :out_layer.bias
Missing key(s) in state_dict :proj_l.weight
Missing key(s) in state_dict :proj_a.weight
Missing key(s) in state_dict :proj_v.weight
Missing key(s) in state_dict :out_layer.weight
Missing key(s) in state_dict :out_layer.bias
[PASS] only moe* tensors are new  (126 new tensors)
       baseline trainable 88,141 | MoE trainable 141,099 (new moe* 52,958) | frozen 1,061,280
[PASS] backbone stays frozen (1,061,280 parameters)  (1,061,280)
[PASS] all moe* parameters are trainable
Missing key(s) in state_dict :proj_l.weight
Missing key(s) in state_dict :proj_a.weight
Missing key(s) in state_dict :proj_v.weight
Missing key(s) in state_dict :out_layer.weight
Missing key(s) in state_dict :out_layer.bias
[PASS] MoE off == baseline PromptModel (all 7 modes)  (max diff 0.00e+00)
[P

,dataset,variant,Avg. ACC,Avg. F1
0,mosi,baseline,62.22,58.23
1,mosi,oracle,75.12,74.72
2,mosi,zero,59.26,50.72
0,sims,baseline,74.58,70.48
1,sims,oracle,80.74,80.44
2,sims,zero,74.00,67.56
0,iemocap,baseline,56.19,53.62
1,iemocap,oracle,66.41,65.68
2,iemocap,zero,52.43,48.42
0,mosei,baseline,74.85,69.25


## 2. Training: 2 runs × 4 datasets (about 1.5 h in total)

Approximate time per run (moe / moe + rec + kd): MOSI 2 / 3 min, CH-SIMS 8 / 12 min, IEMOCAP 8 / 12 min, MOSEI 20 / 30 min.
Runs already in `results/runs.csv` are skipped, so re-running this cell after an interruption continues where it stopped.

In [3]:
for ds in DATASETS:
    for moe, rec, kd in GRID:
        name = run_name(ds, moe, rec, kd)
        if done(name):
            print("[skip]", name)
            continue
        run("train_moe.py", "--dataset", ds, "--moe", moe, "--lambda_rec", rec,
            "--lambda_kd", kd, "--seed", SEED)

[skip] mosi_moe_k3_rec0_kd0_seed1111
[skip] mosi_moe_k3_rec0.5_kd0.5_seed1111
[skip] sims_moe_k3_rec0_kd0_seed1111
[skip] sims_moe_k3_rec0.5_kd0.5_seed1111
[skip] iemocap_moe_k3_rec0_kd0_seed1111
[skip] iemocap_moe_k3_rec0.5_kd0.5_seed1111
[skip] mosei_moe_k3_rec0_kd0_seed1111
=== mosei_moe_k3_rec0.5_kd0.5_seed1111 ===
Missing key(s) in state_dict :proj_l.weight
Missing key(s) in state_dict :proj_a.weight
Missing key(s) in state_dict :proj_v.weight
Missing key(s) in state_dict :out_layer.weight
Missing key(s) in state_dict :out_layer.bias
trainable parameters: 159,243  (new moe*: 63,542)
epoch   1 | 221.3s | task 0.8440 rec 0.6977 kd 0.2949 bal 0.0759 | valid 0.7668 * | gates [-0.141, 0.138, 0.177]
epoch   2 | 225.2s | task 0.8221 rec 0.4521 kd 0.1876 bal 0.1162 | valid 0.7474 * | gates [-0.161, 0.362, 0.17]
epoch   3 | 191.9s | task 0.7962 rec 0.4107 kd 0.2643 bal 0.0322 | valid 0.7421 * | gates [-0.157, 0.398, 0.177]
epoch   4 | 189.2s | task 0.7774 rec 0.4142 kd 0.2938 bal 0.0155 | 

## 3. Final table (test)

Rows per dataset: **Baseline** (`MPLMM/results/table_main_sims_fixed.csv`), **RAG** (K selected on validation, `RAGPrompt/results/final_rag_table.csv`), and the two MoE runs. Same evaluation conventions everywhere.

In [4]:
PROJECT = ROOT.parent
baseline = pd.read_csv(PROJECT / "MPLMM" / "results" / "table_main_sims_fixed.csv").set_index("Dataset")
rag = pd.read_csv(PROJECT / "RAGPrompt" / "results" / "final_rag_table.csv")
rag = rag[rag["selected"] == "✓"].set_index("Dataset")
runs = pd.read_csv(RUNS_CSV)


def scores(source, key, prefix=""):
    """{case: (ACC, F1)} from a row of one of the tables."""
    row = source.loc[key]
    return {c: (row[f"{prefix}{c} ACC"], row[f"{prefix}{c} F1"]) for c in CASES}


results = {}  # (dataset, method) -> {case: (ACC, F1)}
for ds in DATASETS:
    name = NAMES[ds]
    results[(name, "Baseline")] = scores(baseline, name)
    if name in rag.index:
        results[(name, f"RAG (K={int(rag.loc[name, 'K'])})")] = scores(rag, name)
    for moe, rec, kd in GRID:
        label = "MoE + rec + kd" if rec or kd else "MoE"
        match = runs[runs["run"] == run_name(ds, moe, rec, kd)]
        if len(match):
            results[(name, label)] = scores(match.set_index("run"), match["run"].iloc[0], "test ")

final = pd.DataFrame({
    key: {c: f"{acc:.2f} / {f1:.2f}" for c, (acc, f1) in res.items()} for key, res in results.items()
}).T
final.index.names = ["Dataset", "Method"]
final.columns = [f"{c} ACC / F1" for c in CASES]
final.to_csv(ROOT / "results" / "final_moe_table.csv")
final

{a} ACC / F1   {v} ACC / F1   {t} ACC / F1  \
Dataset Method                                                        
MOSI    Baseline        47.23 / 36.99  48.69 / 40.30  75.07 / 75.08   
        RAG (K=10)      53.94 / 54.04  56.85 / 56.78  72.45 / 72.37   
        MoE             58.31 / 57.69  54.96 / 53.66  71.72 / 71.70   
        MoE + rec + kd  54.23 / 53.60  50.87 / 46.15  72.59 / 72.25   
CH-SIMS Baseline        67.40 / 60.68  71.77 / 63.01  78.56 / 78.37   
        RAG (K=5)       66.96 / 59.39  76.15 / 73.17  77.90 / 77.04   
        MoE             67.40 / 60.44  69.15 / 69.46  78.34 / 78.40   
        MoE + rec + kd  70.24 / 58.82  72.43 / 65.31  80.31 / 80.00   
IEMOCAP Baseline        49.24 / 45.19  44.89 / 37.99  57.20 / 57.44   
        RAG (K=5)       52.08 / 50.35  55.87 / 54.25  58.71 / 58.48   
        MoE             53.60 / 50.90  56.44 / 53.17  57.95 / 57.44   
        MoE + rec + kd  51.14 / 47.30  52.08 / 48.47  58.14 / 58.49   
MOSEI   Baseline        71.05 / 59.09  70.92 / 61.29  78.70 / 78.58   
        RAG (K=10)      70.23 / 67.66  69.63 / 65.09  76.03 / 76.71   
        MoE             71.03 / 62.38  62.16 / 63.01  77.17 / 77.76   
        MoE + rec + kd  70.82 / 62.01  68.60 / 64.46  79.09 / 79.06   

                       {a,v} ACC / F1 {a,t} ACC / F1 {v,t} ACC / F1  \
Dataset Method                                                        
MOSI    Baseline        50.87 / 45.63  75.95 / 75.90  75.51 / 75.49   
        RAG (K=10)      46.21 / 33.62  73.32 / 73.40  73.47 / 73.54   
        MoE             60.50 / 60.59  69.83 / 69.91  71.72 / 71.78   
        MoE + rec + kd  50.87 / 45.19  72.01 / 70.35  55.39 / 51.67   
CH-SIMS Baseline        70.02 / 61.31  79.65 / 79.63  80.09 / 79.85   
        RAG (K=5)       73.52 / 69.32  77.46 / 76.78  78.34 / 76.90   
        MoE             71.77 / 72.39  78.12 / 78.03  80.53 / 80.47   
        MoE + rec + kd  71.99 / 63.72  80.31 / 79.32  80.53 / 79.58   
IEMOCAP Baseline        62.12 / 58.83  62.31 / 61.35  61.36 / 60.90   
        RAG (K=5)       64.58 / 63.95  64.02 / 62.86  62.50 / 61.31   
        MoE             66.67 / 65.92  59.47 / 58.55  63.45 / 62.57   
        MoE + rec + kd  60.04 / 57.13  64.20 / 63.64  67.42 / 66.66   
MOSEI   Baseline        71.12 / 59.55  78.16 / 77.97  79.17 / 79.04   
        RAG (K=10)      70.71 / 66.63  75.94 / 76.66  77.15 / 77.67   
        MoE             63.92 / 64.97  77.84 / 78.23  76.61 / 77.29   
        MoE + rec + kd  70.67 / 63.92  79.04 / 79.12  78.81 / 78.69   

                        Avg. ACC / F1  
Dataset Method                         
MOSI    Baseline        62.22 / 58.23  
        RAG (K=10)      62.71 / 60.62  
        MoE             64.50 / 64.22  
        MoE + rec + kd  59.33 / 56.53  
CH-SIMS Baseline        74.58 / 70.48  
        RAG (K=5)       75.05 / 72.10  
        MoE             74.22 / 73.20  
        MoE + rec + kd  75.97 / 71.12  
IEMOCAP Baseline        56.19 / 53.62  
        RAG (K=5)       59.63 / 58.53  
        MoE             59.60 / 58.09  
        MoE + rec + kd  58.84 / 56.95  
MOSEI   Baseline        74.85 / 69.25  
        RAG (K=10)      73.28 / 71.74  
        MoE             71.46 / 70.61  
        MoE + rec + kd  74.50 / 71.21

## 4. Difference tables

**4a.** Each method minus the **baseline**, per case (ACC / F1).
**4b.** Avg. against the **paper** ("Ours", Avg. only).

In [5]:
diff_rows = {}
for (name, method), res in results.items():
    if method == "Baseline":
        continue
    base = results[(name, "Baseline")]
    diff_rows[(name, method)] = {
        f"{c} Δ ACC / F1": f"{res[c][0] - base[c][0]:+.2f} / {res[c][1] - base[c][1]:+.2f}" for c in CASES
    }
diff = pd.DataFrame(diff_rows).T
diff.index.names = ["Dataset", "Method"]
diff

{a} Δ ACC / F1   {v} Δ ACC / F1 {t} Δ ACC / F1  \
Dataset Method                                                            
MOSI    RAG (K=10)       +6.71 / +17.05   +8.16 / +16.48  -2.62 / -2.71   
        MoE             +11.08 / +20.70   +6.27 / +13.36  -3.35 / -3.38   
        MoE + rec + kd   +7.00 / +16.61    +2.18 / +5.85  -2.48 / -2.83   
CH-SIMS RAG (K=5)         -0.44 / -1.29   +4.38 / +10.16  -0.66 / -1.33   
        MoE               +0.00 / -0.24    -2.62 / +6.45  -0.22 / +0.03   
        MoE + rec + kd    +2.84 / -1.86    +0.66 / +2.30  +1.75 / +1.63   
IEMOCAP RAG (K=5)         +2.84 / +5.16  +10.98 / +16.26  +1.51 / +1.04   
        MoE               +4.36 / +5.71  +11.55 / +15.18  +0.75 / +0.00   
        MoE + rec + kd    +1.90 / +2.11   +7.19 / +10.48  +0.94 / +1.05   
MOSEI   RAG (K=10)        -0.82 / +8.57    -1.29 / +3.80  -2.67 / -1.87   
        MoE               -0.02 / +3.29    -8.76 / +1.72  -1.53 / -0.82   
        MoE + rec + kd    -0.23 / +2.92    -2.32 / +3.17  +0.39 / +0.48   

                       {a,v} Δ ACC / F1 {a,t} Δ ACC / F1 {v,t} Δ ACC / F1  \
Dataset Method                                                              
MOSI    RAG (K=10)       -4.66 / -12.01    -2.63 / -2.50    -2.04 / -1.95   
        MoE              +9.63 / +14.96    -6.12 / -5.99    -3.79 / -3.71   
        MoE + rec + kd    +0.00 / -0.44    -3.94 / -5.55  -20.12 / -23.82   
CH-SIMS RAG (K=5)         +3.50 / +8.01    -2.19 / -2.85    -1.75 / -2.95   
        MoE              +1.75 / +11.08    -1.53 / -1.60    +0.44 / +0.62   
        MoE + rec + kd    +1.97 / +2.41    +0.66 / -0.31    +0.44 / -0.27   
IEMOCAP RAG (K=5)         +2.46 / +5.12    +1.71 / +1.51    +1.14 / +0.41   
        MoE               +4.55 / +7.09    -2.84 / -2.80    +2.09 / +1.67   
        MoE + rec + kd    -2.08 / -1.70    +1.89 / +2.29    +6.06 / +5.76   
MOSEI   RAG (K=10)        -0.41 / +7.08    -2.22 / -1.31    -2.02 / -1.37   
        MoE               -7.20 / +5.42    -0.32 / +0.26    -2.56 / -1.75   
        MoE + rec + kd    -0.45 / +4.37    +0.88 / +1.15    -0.36 / -0.35   

                       Avg. Δ ACC / F1  
Dataset Method                          
MOSI    RAG (K=10)       +0.49 / +2.39  
        MoE              +2.28 / +5.99  
        MoE + rec + kd   -2.89 / -1.70  
CH-SIMS RAG (K=5)        +0.47 / +1.62  
        MoE              -0.36 / +2.72  
        MoE + rec + kd   +1.39 / +0.64  
IEMOCAP RAG (K=5)        +3.44 / +4.91  
        MoE              +3.41 / +4.47  
        MoE + rec + kd   +2.65 / +3.33  
MOSEI   RAG (K=10)       -1.57 / +2.49  
        MoE              -3.39 / +1.36  
        MoE + rec + kd   -0.35 / +1.96

In [6]:
PAPER_AVG = {  # MPLMM paper, "Ours", Avg. ACC / F1
    "MOSI": (72.14, 72.57), "IEMOCAP": (67.42, 67.22), "CH-SIMS": (72.07, 77.42), "MOSEI": (73.75, 74.68),
}
paper = pd.DataFrame([
    {"Dataset": name, "Method": method,
     "Avg. ACC": round(res["Avg."][0], 2), "Avg. F1": round(res["Avg."][1], 2),
     "Paper ACC": PAPER_AVG[name][0], "Paper F1": PAPER_AVG[name][1],
     "Δ ACC vs paper": round(res["Avg."][0] - PAPER_AVG[name][0], 2),
     "Δ F1 vs paper": round(res["Avg."][1] - PAPER_AVG[name][1], 2)}
    for (name, method), res in results.items()
]).set_index(["Dataset", "Method"])
paper

Avg. ACC  Avg. F1  Paper ACC  Paper F1  \
Dataset Method                                                   
MOSI    Baseline           62.22    58.23      72.14     72.57   
        RAG (K=10)         62.71    60.62      72.14     72.57   
        MoE                64.50    64.22      72.14     72.57   
        MoE + rec + kd     59.33    56.53      72.14     72.57   
CH-SIMS Baseline           74.58    70.48      72.07     77.42   
        RAG (K=5)          75.05    72.10      72.07     77.42   
        MoE                74.22    73.20      72.07     77.42   
        MoE + rec + kd     75.97    71.12      72.07     77.42   
IEMOCAP Baseline           56.19    53.62      67.42     67.22   
        RAG (K=5)          59.63    58.53      67.42     67.22   
        MoE                59.60    58.09      67.42     67.22   
        MoE + rec + kd     58.84    56.95      67.42     67.22   
MOSEI   Baseline           74.85    69.25      73.75     74.68   
        RAG (K=10)         73.28    71.74      73.75     74.68   
        MoE                71.46    70.61      73.75     74.68   
        MoE + rec + kd     74.50    71.21      73.75     74.68   

                        Δ ACC vs paper  Δ F1 vs paper  
Dataset Method                                         
MOSI    Baseline                 -9.92         -14.34  
        RAG (K=10)               -9.43         -11.95  
        MoE                      -7.64          -8.35  
        MoE + rec + kd          -12.81         -16.04  
CH-SIMS Baseline                  2.51          -6.94  
        RAG (K=5)                 2.98          -5.32  
        MoE                       2.15          -4.22  
        MoE + rec + kd            3.90          -6.30  
IEMOCAP Baseline                -11.23         -13.60  
        RAG (K=5)                -7.79          -8.69  
        MoE                      -7.82          -9.13  
        MoE + rec + kd           -8.58         -10.27  
MOSEI   Baseline                  1.10          -5.43  
        RAG (K=10)               -0.47          -2.94  
        MoE                      -2.29          -4.07  
        MoE + rec + kd            0.75          -3.47

## 5. Are the experts used? (test split)

`gates`: tanh(gate) per generated modality (t, a, v); 0 = MoE switched off. Routing: mean weight per expert and entropy per case (uniform = ln 3 = 1.099; lower = different samples go to different experts).

In [7]:
print(runs[runs["moe"] == 1][["run", "gates", "best_epoch"]].to_string(index=False))
for ds in DATASETS:
    path = ROOT / "results" / "routing" / f"{run_name(ds, 1, 0.5, 0.5)}.csv"
    if path.exists():
        print(f"\n{NAMES[ds]} (moe + rec + kd)")
        display(pd.read_csv(path))

                                 run                    gates  best_epoch
       mosi_moe_k3_rec0_kd0_seed1111  [-0.108, -0.105, 0.104]          29
   mosi_moe_k3_rec0.5_kd0.5_seed1111   [-0.159, 0.097, 0.085]          29
       sims_moe_k3_rec0_kd0_seed1111   [-0.028, 0.052, 0.091]          19
   sims_moe_k3_rec0.5_kd0.5_seed1111  [-0.142, -0.078, 0.108]          23
    iemocap_moe_k3_rec0_kd0_seed1111 [-0.067, -0.003, -0.087]           8
iemocap_moe_k3_rec0.5_kd0.5_seed1111  [-0.097, -0.052, 0.037]          18
      mosei_moe_k3_rec0_kd0_seed1111  [-0.172, -0.008, 0.138]          30
  mosei_moe_k3_rec0.5_kd0.5_seed1111   [-0.199, 0.575, 0.168]          24

MOSI (moe + rec + kd)


,case,generated,mean p1,mean p2,mean p3,entropy,top1 share,top2 share,top3 share
0,{a},t,0.346,0.410,0.244,1.064,0.437,0.563,0.000
1,{a},v,0.370,0.141,0.488,0.988,0.000,0.000,1.000
2,{v},t,0.311,0.273,0.417,1.080,0.007,0.001,0.991
3,{v},a,0.333,0.327,0.339,1.096,0.061,0.340,0.599
4,{t},a,0.318,0.314,0.368,0.923,0.054,0.391,0.555
5,{t},v,0.317,0.323,0.360,0.962,0.402,0.507,0.090
6,"{a,v}",t,0.354,0.292,0.354,1.082,0.487,0.207,0.306
7,"{a,t}",v,0.363,0.281,0.356,0.894,0.449,0.385,0.166
8,"{v,t}",a,0.258,0.440,0.302,0.817,0.006,0.548,0.446



CH-SIMS (moe + rec + kd)


,case,generated,mean p1,mean p2,mean p3,entropy,top1 share,top2 share,top3 share
0,{a},t,0.339,0.420,0.242,1.073,0.000,1.000,0.000
1,{a},v,0.397,0.355,0.248,1.072,1.000,0.000,0.000
2,{v},t,0.298,0.441,0.261,1.068,0.007,0.991,0.002
3,{v},a,0.277,0.310,0.413,1.082,0.000,0.000,1.000
4,{t},a,0.493,0.250,0.257,1.043,1.000,0.000,0.000
5,{t},v,0.328,0.302,0.370,1.092,0.217,0.035,0.748
6,"{a,v}",t,0.532,0.043,0.425,0.817,0.720,0.000,0.280
7,"{a,t}",v,0.352,0.335,0.312,1.093,0.492,0.313,0.195
8,"{v,t}",a,0.434,0.300,0.266,1.069,0.976,0.020,0.004



IEMOCAP (moe + rec + kd)


,case,generated,mean p1,mean p2,mean p3,entropy,top1 share,top2 share,top3 share
0,{a},t,0.255,0.342,0.404,0.928,0.093,0.456,0.451
1,{a},v,0.264,0.442,0.294,1.005,0.025,0.633,0.343
2,{v},t,0.355,0.240,0.405,1.075,0.000,0.000,1.000
3,{v},a,0.297,0.335,0.367,1.088,0.000,0.415,0.585
4,{t},a,0.366,0.341,0.294,1.063,0.644,0.057,0.299
5,{t},v,0.379,0.266,0.354,1.070,0.763,0.072,0.165
6,"{a,v}",t,0.275,0.228,0.497,0.900,0.252,0.140,0.608
7,"{a,t}",v,0.312,0.286,0.402,0.970,0.059,0.339,0.602
8,"{v,t}",a,0.367,0.366,0.266,1.040,0.330,0.430,0.241



MOSEI (moe + rec + kd)


,case,generated,mean p1,mean p2,mean p3,entropy,top1 share,top2 share,top3 share
0,{a},t,0.302,0.345,0.352,1.066,0.244,0.284,0.473
1,{a},v,0.261,0.337,0.402,1.059,0.000,0.418,0.582
2,{v},t,0.335,0.348,0.318,1.037,0.266,0.363,0.371
3,{v},a,0.302,0.418,0.279,1.004,0.509,0.470,0.020
4,{t},a,0.349,0.361,0.290,1.074,0.441,0.554,0.005
5,{t},v,0.370,0.329,0.301,1.041,0.277,0.313,0.410
6,"{a,v}",t,0.352,0.314,0.333,1.033,0.328,0.168,0.504
7,"{a,t}",v,0.314,0.374,0.312,0.988,0.192,0.505,0.303
8,"{v,t}",a,0.315,0.341,0.344,1.030,0.465,0.322,0.213


In [8]:
PROJECT = ROOT.parent
paper = pd.read_csv(ROOT / "results" / "paper_ours.csv")  # Dataset, Case, ACC, F1 (paper "Ours")

base = pd.read_csv(PROJECT / "MPLMM" / "results" / "table_main_sims_fixed.csv").set_index("Dataset")
rag = pd.read_csv(PROJECT / "RAGPrompt" / "results" / "final_rag_table.csv")
rag = rag[rag["selected"] == "✓"].set_index("Dataset")
runs = pd.read_csv(RUNS_CSV)

rows = []
for ds in DATASETS:
    name = NAMES[ds]

    def add(method, get):
        for c in CASES:
            acc, f1 = get(c)
            rows.append({"Dataset": name, "Method": method, "Case": c, "ours ACC": acc, "ours F1": f1})

    add("Baseline", lambda c: (base.loc[name, f"{c} ACC"], base.loc[name, f"{c} F1"]))
    add(f"RAG (K={int(rag.loc[name, 'K'])})", lambda c: (rag.loc[name, f"{c} ACC"], rag.loc[name, f"{c} F1"]))
    for moe, rec, kd in GRID:
        run = runs[runs["run"] == run_name(ds, moe, rec, kd)]
        if len(run):
            r = run.iloc[0]
            label = "MoE + rec + kd" if rec or kd else "MoE"
            add(label, lambda c, r=r: (r[f"test {c} ACC"], r[f"test {c} F1"]))

ours = pd.DataFrame(rows)
cmp_ = ours.merge(paper, on=["Dataset", "Case"], how="left")
cmp_["Δ ACC"] = (cmp_["ours ACC"] - cmp_["ACC"]).round(2)
cmp_["Δ F1"] = (cmp_["ours F1"] - cmp_["F1"]).round(2)

diff = cmp_.pivot_table(index=["Dataset", "Method"], columns="Case",
                        values=["Δ ACC", "Δ F1"], sort=False)
diff = diff.reindex(columns=pd.MultiIndex.from_product([["Δ ACC", "Δ F1"], CASES]))
diff.to_csv(ROOT / "results" / "diff_vs_paper.csv")
diff.round(2)


Δ ACC                                            \
                          {a}    {v}    {t}  {a,v}  {a,t}  {v,t}   Avg.   
Dataset Method                                                            
MOSI    Baseline       -15.48 -14.43  -5.05 -14.15  -4.81  -5.61  -9.92   
        RAG (K=10)      -8.77  -6.27  -7.67 -18.81  -7.44  -7.65  -9.43   
        MoE             -4.40  -8.16  -8.40  -4.52 -10.93  -9.40  -7.64   
        MoE + rec + kd  -8.48 -12.25  -7.53 -14.15  -8.75 -25.73 -12.81   
CH-SIMS Baseline         1.47   5.75  -1.19   4.74   2.20   2.12   2.51   
        MoE              1.47   3.13  -1.41   6.49   0.67   2.56   2.15   
        MoE + rec + kd   4.31   6.41   0.56   6.71   2.86   2.56   3.90   
        RAG (K=5)        1.03  10.13  -1.85   8.24   0.01   0.37   2.98   
IEMOCAP Baseline       -10.53 -12.72 -12.03  -5.14 -13.67 -13.32 -11.23   
        MoE             -6.17  -1.17 -11.28  -0.59 -16.51 -11.23  -7.82   
        MoE + rec + kd  -8.63  -5.53 -11.09  -7.22 -11.78  -7.26  -8.58   
        RAG (K=5)       -7.69  -1.74 -10.52  -2.68 -11.96 -12.18  -7.79   
MOSEI   Baseline         3.72   3.63  -0.42   2.91  -2.29  -0.94   1.10   
        RAG (K=10)       2.90   2.34  -3.09   2.50  -4.51  -2.96  -0.47   
        MoE              3.70  -5.13  -1.95  -4.29  -2.61  -3.50  -2.29   
        MoE + rec + kd   3.49   1.31  -0.03   2.46  -1.41  -1.30   0.75   

                         Δ F1                                            
                          {a}    {v}    {t}  {a,v}  {a,t}  {v,t}   Avg.  
Dataset Method                                                           
MOSI    Baseline       -26.66 -23.44  -5.23 -19.78  -5.19  -5.70 -14.34  
        RAG (K=10)      -9.61  -6.96  -7.94 -31.79  -7.69  -7.65 -11.95  
        MoE             -5.96 -10.08  -8.61  -4.82 -11.18  -9.41  -8.35  
        MoE + rec + kd -10.05 -17.59  -8.06 -20.22 -10.74 -29.52 -16.04  
CH-SIMS Baseline       -16.42 -15.85  -0.37 -12.71   1.79   1.90  -6.94  
        MoE            -16.66  -9.40  -0.34  -1.63   0.19   2.52  -4.22  
        MoE + rec + kd -18.28 -13.55   1.26 -10.30   1.48   1.63  -6.30  
        RAG (K=5)      -17.71  -5.69  -1.70  -4.70  -1.06  -1.05  -5.32  
IEMOCAP Baseline       -14.52 -18.99 -11.84  -8.54 -14.09 -13.61 -13.60  
        MoE             -8.81  -3.81 -11.84  -1.45 -16.89 -11.94  -9.13  
        MoE + rec + kd -12.41  -8.51 -10.79 -10.24 -11.80  -7.85 -10.27  
        RAG (K=5)       -9.36  -2.73 -10.80  -3.42 -12.58 -13.20  -8.69  
MOSEI   Baseline        -9.62  -8.11  -0.59 -10.36  -2.46  -1.09  -5.43  
        RAG (K=10)      -1.05  -4.31  -2.46  -3.28  -3.77  -2.46  -2.94  
        MoE             -6.33  -6.39  -1.41  -4.94  -2.20  -2.84  -4.07  
        MoE + rec + kd  -6.70  -4.94  -0.11  -5.99  -1.31  -1.44  -3.47